# Load dataset

In [4]:
import pandas as pd
import numpy as np

import spacy
from sklearn.feature_extraction.text import CountVectorizer


# POS and NER with spaCy

In [6]:
# Load dataset
df = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
import spacy
nlp = spacy.load("en_core_web_sm")

def get_pos(text):
    doc = nlp(text)
    return [(token.text, token.pos_) for token in doc]

def get_ner(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

# Apply to a small sample for speed
sample = df.head(10)
sample['pos'] = sample['review'].apply(get_pos)
sample['ner'] = sample['review'].apply(get_ner)
sample[['review','pos','ner']]


/tmp/ipykernel_58/3690725613.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample['pos'] = sample['review'].apply(get_pos)
/tmp/ipykernel_58/3690725613.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample['ner'] = sample['review'].apply(get_ner)


,review,pos,ner
0,One of the other reviewers has mentioned that ...,"[(One, NUM), (of, ADP), (the, DET), (other, AD...","[(One, CARDINAL), (1, CARDINAL), (GO, ORG), (O..."
1,A wonderful little production. <br /><br />The...,"[(A, DET), (wonderful, ADJ), (little, ADJ), (p...","[(BBC, ORG), (Michael Sheen, PERSON), (William..."
2,I thought this was a wonderful way to spend ti...,"[(I, PRON), (thought, VERB), (this, PRON), (wa...","[(summer weekend, DATE), (Match Point 2, PERSO..."
3,Basically there's a family where a little boy ...,"[(Basically, ADV), (there, PRON), ('s, VERB), ...","[(Jake, NORP), (Jake, NORP), (Rambo, PERSON), ..."
4,"Petter Mattei's ""Love in the Time of Money"" is...","[(Petter, PROPN), (Mattei, PROPN), ('s, PART),...","[(Love, WORK_OF_ART), (Mattei, PERSON), (Arthu..."
5,"Probably my all-time favorite movie, a story o...","[(Probably, ADV), (my, PRON), (all, DET), (-, ...","[(15, CARDINAL), (the last 25 years, DATE), (P..."
6,I sure would like to see a resurrection of a u...,"[(I, PRON), (sure, ADV), (would, AUX), (like, ...","[(today, DATE), (every week, DATE), (10, CARDI..."
7,"This show was an amazing, fresh & innovative i...","[(This, DET), (show, NOUN), (was, AUX), (an, D...","[(70, DATE), (first, ORDINAL), (first, ORDINAL..."
8,Encouraged by the positive comments about this...,"[(Encouraged, VERB), (by, ADP), (the, DET), (p...","[(950+, DATE), (one, CARDINAL), (no less than ..."
9,If you like original gut wrenching laughter yo...,"[(If, SCONJ), (you, PRON), (like, ADP), (origi...",[]


# BoW (no stop words yet)

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X_bow = cv.fit_transform(df['review'])

print("BoW vocabulary size:", len(cv.get_feature_names_out()))


BoW vocabulary size: 101895


# BoW with stop words

In [10]:
cv_sw = CountVectorizer(stop_words='english')
X_bow_sw = cv_sw.fit_transform(df['review'])

print("BoW (stopwords removed) vocab size:", len(cv_sw.get_feature_names_out()))


BoW (stopwords removed) vocab size: 101583


# N‑gram features (unigram + bigram)

In [11]:
cv_ngram = CountVectorizer(ngram_range=(1,2), stop_words='english')
X_ngram = cv_ngram.fit_transform(df['review'])

print("N‑gram (1–2, stopwords removed) vocab size:", len(cv_ngram.get_feature_names_out()))


N‑gram (1–2, stopwords removed) vocab size: 3081097


# Inspect and compare the feature spaces

In [12]:
print("BoW vocab:", len(cv.get_feature_names_out()))
print("BoW (stopwords removed) vocab:", len(cv_sw.get_feature_names_out()))
print("N‑gram vocab:", len(cv_ngram.get_feature_names_out()))


BoW vocab: 101895
BoW (stopwords removed) vocab: 101583
N‑gram vocab: 3081097


# Visualize the most frequent words

In [13]:
import numpy as np

word_counts = np.asarray(X_bow_sw.sum(axis=0)).ravel()
vocab = cv_sw.get_feature_names_out()

freq_df = pd.DataFrame({'word': vocab, 'count': word_counts})
freq_df.sort_values('count', ascending=False).head(20)


,word,count
11989,br,201951
60055,movie,87971
33134,film,79705
52602,like,40172
48334,just,35184
37983,good,29753
90676,time,25110
85982,story,23119
73161,really,23094
7551,bad,18473
